# Colab: LoRA fine-tuning for Component 1

**J26-SE-325** — run this on Colab, not on the development laptop.

Two independent reasons the local machine cannot do this:

1. **No CUDA.** The dev machine has an AMD Radeon 610M, so `torch` there is the CPU build.
   Fine-tuning a 200M-parameter model on CPU is hours per run.
2. **~24 KB/s network.** The foundation checkpoints are ~821 MB each — roughly 10 hours
   apiece. Three attempts left only zero-byte `.incomplete` placeholders.

Colab solves both: a free T4 and a fast link.

## What this notebook does

| Part | Output |
|---|---|
| A | LoRA-fine-tune TimesFM and/or Chronos-Bolt → adapter weights (a few MB) |
| B | SFT-fine-tune a small Llama on the tool-calling trajectories → the Phase 5c agent |

Both produce artifacts small enough to bring home over a slow link. **Download the adapters,
not the base models** — that asymmetry is the entire point of using LoRA here.

## Before running

Runtime → Change runtime type → **T4 GPU**.
## Running this in Google Colab

Upload or open this notebook on its own, then **Runtime > Run all**. The first code
cell clones the `Nivakaran` branch, installs what this notebook needs, and chdirs into
`backend/Portfolio-Optimization`. Nothing else has to be uploaded alongside it.

That cell is safe to re-run, and on a local checkout it installs nothing -- it only
locates the component root, so the same notebook works in both places.

**Set the runtime to a GPU before running** (Runtime > Change runtime type > T4 GPU).
The install pulls the full ML stack, so the first cell takes a few minutes.


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> T4 GPU'

## Setup

Clone the repo, install only what fine-tuning needs. `timesfm` is pinned at **2.0.2** —
there is no `2.5` on PyPI; "2.5" is the model generation exposed as
`TimesFM_2p5_200M_torch`. Installing `2.5` fails outright.

In [ ]:
# === BOOTSTRAP =========================================================================
# Makes this notebook runnable on its own: open it in Colab, run this cell, run the rest.
# Safe to re-run at any point -- it never clones twice and never nests a checkout, and it
# brings an existing checkout up to the branch head rather than leaving it stale.
# On a local checkout it only locates the component root; it installs nothing.

REPO_URL = "https://github.com/SE-Y4S1/J26-SE-325.git"
BRANCH = "Nivakaran"  # the component is NOT on main
SUBDIR = "backend/Portfolio-Optimization"
PIP_PKGS = "timesfm[torch]==2.0.2 chronos-forecasting peft accelerate pandas-ta-openbb yfinance pymoo scikit-fuzzy mlflow pydantic pyyaml pyarrow statsmodels"

import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

MARKERS = ("configs/resolved_universe.yaml", "optimization")
REQUIRED = ['configs/resolved_universe.yaml', 'forecasting/finetune_lora.py', 'features/feature_store.py', 'data/window_selector.py']
IMPORTS = ['torch', 'peft', 'pandas', 'numpy']

# Every top-level package this component owns. Used to drop stale module objects after the
# checkout changes underneath a kernel that has already imported some of them.
OWN_PACKAGES = {
    "agent", "configs", "data", "evaluation", "experiments",
    "features", "forecasting", "optimization", "service",
}


def find_root(start):
    """The component root is the nearest ancestor holding every marker path."""
    for base in (start, *start.parents):
        if all((base / m).exists() for m in MARKERS):
            return base
    return None


def run(cmd):
    return subprocess.run(cmd).returncode == 0


root = find_root(Path.cwd())

if root is None:
    if not IN_COLAB:
        raise SystemExit(
            "Not inside a checkout of backend/Portfolio-Optimization, and not running "
            "on Colab. cd into the component directory, then restart the kernel."
        )
    # Absolute target path on purpose: re-running this cell after the os.chdir below
    # must not clone a second copy inside the first one.
    checkout = Path("/content/J26-SE-325")

    if (checkout / SUBDIR).exists():
        # A session that ran this cell earlier already has a checkout, pinned to whenever
        # it first ran. Leaving it alone means a fix pushed since then looks like it was
        # never applied -- the symptom is a ModuleNotFoundError for a file that exists on
        # the branch. Bring it to the branch head instead.
        print("Updating existing checkout to the head of " + BRANCH + " ...")
        if run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", BRANCH]):
            run(["git", "-C", str(checkout), "reset", "--hard", "FETCH_HEAD"])
            run(["git", "-C", str(checkout), "clean", "-fd"])
        else:
            print("  fetch failed; continuing with the checkout already on disk.")
            print("  If a later cell reports a missing module, delete it and re-run:")
            print("    !rm -rf /content/J26-SE-325")
    else:
        print("Cloning branch " + BRANCH + " ...")
        clone = ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)]
        if not run(clone):
            # Private repo, or the branch is missing. Ask, rather than leaving git to
            # block on a credential prompt that Colab has no way to answer.
            from getpass import getpass

            token = getpass("Clone failed. GitHub token (repo scope), blank to abort: ").strip()
            if not token:
                raise SystemExit("Cannot reach " + REPO_URL + " on branch " + BRANCH + ".")
            authed = REPO_URL.replace("https://", "https://" + token + "@")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", BRANCH, authed, str(checkout)],
                check=True,
            )
    root = checkout / SUBDIR

os.chdir(root)
if str(root) not in sys.path:
    # Set here rather than in a later cell, so the cells can be run out of order.
    sys.path.insert(0, str(root))

# The files on disk may have just changed. Drop any already-imported project modules and
# invalidate the import caches, or this kernel keeps serving the code it read the first
# time -- including a package whose new submodule it will insist does not exist.
for _name in [n for n in sys.modules if n.split(".")[0] in OWN_PACKAGES]:
    del sys.modules[_name]
importlib.invalidate_caches()

if IN_COLAB and PIP_PKGS:
    print("Installing dependencies ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_PKGS.split(), check=True)

# Verify the checkout before anything imports from it. A missing module three cells from
# now reads like a model problem; caught here it reads like the checkout problem it is.
missing = [m for m in REQUIRED if not (root / m).exists()]
if missing:
    raise SystemExit(
        "Wrong or outdated checkout at " + str(root) + " -- missing " + repr(missing)
        + ". Delete it and re-run this cell:  !rm -rf /content/J26-SE-325"
    )

failed = []
for mod in IMPORTS:
    try:
        __import__(mod)
    except ImportError as exc:
        failed.append(mod + " (" + str(exc) + ")")
if failed:
    raise SystemExit(
        "Imports failed after install: "
        + ", ".join(failed)
        + ". If Colab replaced a pre-installed package, use Runtime -> Restart session, "
        "then re-run this cell."
    )

# Generate the capability report. forecasting/base.py decides which foundation adapters
# to register by reading artifacts/env_report.json -- and artifacts/ is gitignored, so a
# fresh clone has none and every foundation model reports itself "not installed" however
# the install actually went. Without this step that failure surfaces many cells later, as
# a message blaming the package rather than the missing report.
try:
    from forecasting.env_report import write_env_report
except ModuleNotFoundError as exc:
    raise SystemExit(
        "Checkout is behind the branch (" + str(exc) + "). Delete it and re-run this "
        "cell:  !rm -rf /content/J26-SE-325"
    ) from exc

report = write_env_report()
forecasters = report["optional_forecasters"]

print("OK  root=" + str(root))
print("    python=" + sys.version.split()[0] + "  colab=" + str(IN_COLAB))
print("    torch=" + str(report["torch"]) + "  cuda=" + str(report["torch_cuda_available"]))
for fname in sorted(forecasters):
    meta = forecasters[fname]
    if meta.get("available"):
        print("    " + fname + ": available (version " + str(meta.get("version")) + ")")
    else:
        print("    " + fname + ": UNAVAILABLE -- " + str(meta.get("error", "no reason recorded")))

if not any(m.get("available") for m in forecasters.values()):
    print()
    print("    No foundation model imported. RQ1 foundation/hybrid rows cannot be produced;")
    print("    everything else in this notebook still runs. If a pip install replaced a")
    print("    pre-installed package, use Runtime > Restart session and re-run this cell.")


In [ ]:
# The bootstrap installed `timesfm[torch]`, which can pull a CPU-only torch wheel over
# Colab's CUDA build. Verify that before spending a session's GPU time finding out on CPU.
import torch

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU visible. Two possible causes: "
        "(1) the runtime is CPU -- Runtime > Change runtime type > T4 GPU, then re-run "
        "from the top; or (2) the runtime IS a GPU one but torch was replaced by a CPU "
        "wheel -- reinstall the CUDA build with: pip install --force-reinstall torch "
        "--index-url https://download.pytorch.org/whl/cu121  then Runtime > Restart session."
    )

print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Resolve the peft <-> torchao conflict before any LoRA work.
#
# When peft builds a LoRA layer it walks a list of dispatchers, one of which asks
# is_torchao_available(). That function does not answer False for an old torchao -- it
# RAISES. Colab ships torchao 0.10.0 and current peft wants >= 0.16.0, so every fine-tune
# died with "Found an incompatible version of torchao" before training started.
#
# The dispatcher exists to handle QUANTIZED layers. This fine-tune uses none, so torchao is
# pure overhead here. Removing it makes is_torchao_available() return False cleanly, which
# is the answer peft actually wanted. Upgrading it instead would drag a torch rebuild onto
# a runtime whose CUDA torch we just finished verifying.
import importlib
import subprocess
import sys

try:
    from importlib.metadata import version

    installed = version("torchao")
except Exception:
    installed = None

if installed is None:
    print("torchao not installed -- nothing to resolve.")
else:
    print("torchao " + installed + " present; peft raises on anything below 0.16.0.")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    importlib.invalidate_caches()
    print("removed torchao (no quantized layers are used in this fine-tune).")

# Prove peft can now reach the point that used to fail: building a LoRA-wrapped module.
for _name in [n for n in list(sys.modules) if n.split(".")[0] in {"peft", "torchao"}]:
    del sys.modules[_name]
importlib.invalidate_caches()

import torch
from peft import LoraConfig, get_peft_model


class _Probe(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.q_proj = torch.nn.Linear(8, 8)

    def forward(self, x):
        return self.q_proj(x)


_probe = get_peft_model(_Probe(), LoraConfig(target_modules=["q_proj"], r=2, bias="none"))
print("peft can build LoRA layers:", sum(p.numel() for p in _probe.parameters() if p.requires_grad),
      "trainable parameters in the probe")
del _probe


## Part A — LoRA fine-tune the foundation forecasters

Builds the same feature table the local pipeline uses, driven by the committed
`configs/resolved_universe.yaml` so each symbol contributes exactly the history its own
criteria justified. Nothing about the windows is re-decided here.

In [ ]:
import sys, warnings, logging
from datetime import date
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
sys.path.insert(0, str(Path.cwd()))

import pandas as pd, yaml
from data.ingestion import bars_to_frame, fetch_ohlcv
from data.schema import AssetClass
from features.feature_store import add_targets, build_feature_table
from features.technical import compute_universe_indicators

CLASSES = {'equity': AssetClass.EQUITY, 'etf': AssetClass.ETF, 'forex': AssetClass.FOREX}
resolved = yaml.safe_load(Path('configs/resolved_universe.yaml').read_text())['symbols']

bars = {}
for symbol, entry in sorted(resolved.items()):
    rows = fetch_ohlcv(symbol, CLASSES[entry['asset_class']],
                       date.fromisoformat(entry['history_start']),
                       date.fromisoformat(entry['history_end']))
    if rows:
        bars[symbol] = bars_to_frame(rows)

HORIZON = 5
features = add_targets(build_feature_table(bars, compute_universe_indicators(bars)), horizon=HORIZON)
print(f'{len(bars)} symbols | {len(features):,} feature rows | '
      f"{features.timestamp.min().date()} -> {features.timestamp.max().date()}")

### Diagnostic: what can this session actually run?

Run this before fine-tuning. It reports whether each foundation model imports, whether
its weights are present, and which Linear modules LoRA would adapt. If fine-tuning
fails later, the answer is almost always visible here first.

In [ ]:
from forecasting.base import available_foundation_models, usable_foundation_models, get_forecaster
from forecasting.finetune_lora import _discover_target_modules, _resolve_inner_model

print('packages importable :', available_foundation_models())
print('weights cached      :', usable_foundation_models())
print()

for name in ('timesfm', 'chronos_bolt'):
    try:
        forecaster = get_forecaster(name)
        forecaster._load()          # downloads weights on first call
        inner = _resolve_inner_model(forecaster)
        targets = _discover_target_modules(inner)
        n_params = sum(p.numel() for p in inner.parameters())
        print(f'{name}: OK | {type(inner).__name__} | {n_params/1e6:.0f}M params')
        print(f'   LoRA would target: {targets}')
    except Exception as exc:
        import traceback
        print(f'{name}: UNUSABLE -> {type(exc).__name__}: {exc}')
        traceback.print_exc()
    print()

### Smoke check before the real run

One epoch on a few hundred rows. Every remaining failure mode in the fine-tuning path -- device, dtype, shapes, quantile columns -- shows up here in about a minute rather than after a twenty-epoch run.


In [ ]:
# Smoke check: one epoch, a few hundred rows, tiny batches.
#
# The full fine-tune below is 20 epochs over the whole feature table, so anything wrong with
# the forward/backward path -- a device mismatch, a dtype mismatch, a shape error, a wrong
# quantile column -- costs a long run before it shows itself. This exercises exactly the
# same code path in about a minute and stops the notebook if it fails.
#
# register=False and log_to_mlflow=False on purpose: a smoke run must not register a
# checkpoint or leave an MLflow run that looks like a real result.
import torch

from forecasting.finetune_lora import LoRAConfig, finetune

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (fine-tuning will be slow)')
print()

smoke_features = features.groupby('symbol', sort=False).head(400)
smoke_config = LoRAConfig(r=4, lora_alpha=8, epochs=1, batch_size=4)

for model_name in ('timesfm', 'chronos_bolt'):
    print(f'--- {model_name} ---')
    finetune(
        model_name, smoke_features, horizon=HORIZON, config=smoke_config,
        output_dir=Path(f'artifacts/smoke/{model_name}'),
        register=False, log_to_mlflow=False,
    )
    print(f'{model_name}: forward and backward OK')
    print()

print('Smoke check passed for both models -- the full fine-tune below should run.')


In [ ]:
import textwrap
import traceback

from forecasting.finetune_lora import UNSUPPORTED_LORA, LoRAConfig, finetune

# Larger than the CPU defaults -- a T4 can afford it.
config = LoRAConfig(r=16, lora_alpha=32, epochs=20, batch_size=64, lr=5e-5)

adapters = {}
skipped = {}

for model_name in ('timesfm', 'chronos_bolt'):
    if model_name in UNSUPPORTED_LORA:
        # A documented limitation, not something to debug, so no traceback for it. The
        # model still earns its zero-shot RQ1 row; only the LoRA row is unavailable.
        skipped[model_name] = UNSUPPORTED_LORA[model_name]
        print(f'SKIP  {model_name}: LoRA not supported for this architecture.')
        print(textwrap.fill(UNSUPPORTED_LORA[model_name], width=88,
                            initial_indent='      ', subsequent_indent='      '))
        print()
        continue

    try:
        adapters[model_name] = finetune(
            model_name, features, horizon=HORIZON, config=config,
            output_dir=Path(f'artifacts/checkpoints/{model_name}-lora-h{HORIZON}'),
        )
        print(f'OK    {model_name} -> {adapters[model_name]}')
    except Exception as exc:
        # One model failing must not cost the other; RQ1 reports whichever succeeded.
        # The full traceback matters here -- LoRA target-module discovery and missing
        # weights fail very differently, and the summary line alone cannot tell them apart.
        print(f'FAILED {model_name}: {type(exc).__name__}: {exc}')
        traceback.print_exc()

print()
print(f'adapters produced: {list(adapters) or "NONE"}')

attempted = [m for m in ('timesfm', 'chronos_bolt') if m not in skipped]
if not attempted:
    print('Every architecture is marked unsupported, so there was nothing to fine-tune.')
elif not adapters:
    print('Every attempted fine-tune failed. Read the tracebacks above -- the next cell')
    print('falls back to a zero-shot base so the notebook can continue, but RQ1 will have')
    print('no LoRA rows at all.')
elif skipped:
    print(f'RQ1 will have LoRA rows for {list(adapters)} and zero-shot rows for everything,')
    print(f'including {list(skipped)}.')

adapters


### Train the hybrid head

The residual head is the part the TAF actually commits to ("hybrid TimesFM + LSTM/MLP").
It is zero-initialised, so the hybrid starts *exactly* at the base model's accuracy and any
improvement is attributable to the covariates rather than to a lucky init.

In [ ]:
from forecasting.base import get_forecaster
from forecasting.hybrid_model import HybridConfig, HybridForecaster
from forecasting.residual_head import ResidualHeadConfig

# Pick a base. If a LoRA adapter exists we use it; if fine-tuning failed we fall back to the
# ZERO-SHOT foundation model rather than stopping. The residual head is trained either way,
# so the hybrid still gets built and RQ1 still gets a hybrid row -- it just sits on an
# un-tuned base, which is a result worth reporting rather than a dead end.
available = [m for m in ('timesfm', 'chronos_bolt') if m in adapters]

if available:
    base_name = available[0]
    adapter_path = str(adapters[base_name])
    print(f'using fine-tuned {base_name} from {adapter_path}')
else:
    print('NO ADAPTERS were produced -- scroll up and read the FAILED lines from the')
    print('fine-tuning cell; that message is the real error.')
    print('Falling back to a ZERO-SHOT base so the hybrid can still be trained.')
    base_name, adapter_path = 'timesfm', None

try:
    base = get_forecaster(base_name, adapter_path=adapter_path)
except Exception as exc:
    print(f'{base_name} unavailable ({exc}); trying the other foundation model')
    base_name = 'chronos_bolt' if base_name == 'timesfm' else 'timesfm'
    base = get_forecaster(base_name, adapter_path=None)

print(f'base model: {base_name} ({"LoRA" if adapter_path else "zero-shot"})')

hybrid = HybridForecaster(base, HybridConfig(
    base_model=base_name,
    head=ResidualHeadConfig(hidden_size=128, num_layers=2, epochs=40),
    window=60,
))
hybrid.fit(features, horizon=HORIZON)

decomposed = hybrid.decompose(features, horizon=HORIZON)
print('base / residual / final (means):')
for q in (10, 50, 90):
    print(f'  p{q}: {decomposed[f"base_p{q}"].mean():+.5f}  '
          f'{decomposed[f"residual_p{q}"].mean():+.5f}  {decomposed[f"final_p{q}"].mean():+.5f}')


### RQ1 — the full comparison

This is the table the local machine cannot produce. Walk-forward, no look-ahead, every model
on identical folds.

In [ ]:
from evaluation.backtest import BacktestConfig, run_walk_forward
from evaluation.metrics import forecast_metrics

config_bt = BacktestConfig(train_window_days=1095, test_window_days=180,
                           step_days=180, embargo_days=HORIZON)

# baseline_lstm always runs -- it trains from scratch and needs no downloaded weights, so
# RQ1 always has at least its comparison point. The rest are attempted and reported either
# way; a model that could not run is stated as such rather than silently omitted, because a
# table with a missing row reads as "not compared" not "not available".
candidates = ['baseline_lstm'] + [m for m in ('timesfm', 'chronos_bolt', 'hybrid')]

rows = []
for name in candidates:
    try:
        if name == 'hybrid':
            preds = run_walk_forward(features, 'hybrid', horizon=HORIZON, config=config_bt,
                                     forecaster_kwargs={'base_model': base_name})
        else:
            preds = run_walk_forward(features, name, horizon=HORIZON, config=config_bt)
        valid = preds.dropna(subset=['target_return'])
        rows.append({'model': name, 'status': 'ok', 'n_folds': int(preds.fold.nunique()),
                     'n_predictions': len(valid),
                     **forecast_metrics(valid.target_return.to_numpy(),
                                        valid[['p10','p50','p90']].to_numpy(), (0.1,0.5,0.9))})
        print(f'OK  {name}')
    except Exception as exc:
        rows.append({'model': name, 'status': f'{type(exc).__name__}: {exc}'})
        print(f'SKIP {name}: {exc}')

rq1 = pd.DataFrame(rows)
Path('artifacts/results').mkdir(parents=True, exist_ok=True)
rq1.to_csv('artifacts/results/rq1_forecast.csv', index=False)
rq1


**Reading the table.** Compare on **pinball loss**, not MAE — MAE only scores the median and
says nothing about whether the p10–p90 band is honest, which is what Phase 5a's CVaR consumes.
Also check `calibration_error_*`: the local baseline showed all three quantiles sitting below
nominal (a uniform level bias from trailing-window training through a bull market). If the
hybrid closes that gap, say so explicitly — it is the clearest evidence the residual head
earns its place.

## Part B — SFT the tool-calling agent

Trains the model that replaces `agent/reference_agent.py`. The dataset is generated locally
by `agent/trajectory_generation.py`; every trajectory in it passed `enforce_grounding`, so
the model learns to route numbers through the optimizer rather than inventing them.

**Upload `artifacts/trajectories/*.jsonl` to the Colab session before running this.**

In [ ]:
import json, glob

paths = sorted(glob.glob('artifacts/trajectories/*.jsonl'))
assert paths, 'No trajectories. Run agent/trajectory_generation.py locally and upload the JSONL.'

records = [json.loads(line) for p in paths for line in open(p, encoding='utf-8')]
print(f'{len(records)} trajectories from {len(paths)} file(s)')

# Non-negotiable: every record must have called the grounded tool. Training on even a few
# ungrounded transcripts teaches the model that inventing a number is sometimes acceptable,
# which is the exact failure the whole design exists to prevent.
GROUNDING_TOOL = 'run_fuzzy_ga_withdrawal'
bad = [r for r in records if GROUNDING_TOOL not in r['metadata']['tool_calls']]
assert not bad, f'{len(bad)} ungrounded trajectories — regenerate, do not train on these'
print('all trajectories grounded')

In [ ]:
!pip install -q transformers trl datasets bitsandbytes 2>&1 | tail -2

BASE_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'   # gated: accept the licence on HF first
# Ungated alternative if the licence is a blocker:
# BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

from huggingface_hub import notebook_login
notebook_login()      # skip if the chosen base is ungated

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(record):
    """Render a trajectory through the model's own chat template.

    Using apply_chat_template rather than hand-formatting matters: the tool-call special
    tokens must match what the model emits at inference, or the fine-tune teaches a format
    the runtime cannot parse.
    """
    messages = []
    for m in record['messages']:
        entry = {'role': m['role'], 'content': m.get('content') or ''}
        if m.get('tool_calls'):
            entry['content'] = json.dumps(m['tool_calls'])
        messages.append(entry)
    return tokenizer.apply_chat_template(messages, tokenize=False)

dataset = Dataset.from_dict({'text': [to_text(r) for r in records]}).train_test_split(test_size=0.1, seed=42)
print(dataset)
print(dataset['train'][0]['text'][:600])

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto', load_in_4bit=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ),
    args=SFTConfig(
        output_dir='artifacts/checkpoints/agent-sft',
        num_train_epochs=3, per_device_train_batch_size=2,
        gradient_accumulation_steps=8, learning_rate=2e-4,
        logging_steps=20, eval_strategy='epoch', save_strategy='epoch',
        bf16=True, max_length=2048, report_to='none',
    ),
)
trainer.train()
trainer.save_model('artifacts/checkpoints/agent-sft')

## Bring the results home

Download **adapters only** — a few MB against ~821 MB of base weights. On a 24 KB/s link
that is the difference between a minute and a day.

In [ ]:
import shutil
from pathlib import Path

shutil.make_archive('component1_adapters', 'zip', 'artifacts/checkpoints')
size_mb = Path('component1_adapters.zip').stat().st_size / 1e6
print(f'component1_adapters.zip  {size_mb:.1f} MB')

from google.colab import files
files.download('component1_adapters.zip')

# Also take the RQ1 table — it is the dissertation result, and it is only kilobytes.
files.download('artifacts/results/rq1_forecast.csv')

## Back on the local machine

```bash
unzip component1_adapters.zip -d artifacts/checkpoints/
```

Then register the checkpoints so `/portfolio/*` responses carry a resolvable
`model_version` — Component 3 anchors provenance on it, and an unregistered adapter is
invisible to the on-chain bridge:

```python
from datetime import date
from pathlib import Path
from forecasting.model_registry import register

register('hybrid-timesfm', Path('artifacts/checkpoints/timesfm-lora-h5'),
         train_start=date(2011, 1, 3), train_end=date(2025, 12, 31),
         metrics={'val_pinball': 0.0},   # copy the real number from the RQ1 table
         activate=True)
```

For the agent, serve the SFT adapter through Ollama (or vLLM) and point
`OLLAMA_MODEL` at it. The tools, schemas and `enforce_grounding` validator stay exactly as
they are — only the driver changes, which is what the reference agent was built to prove.